> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 07 · Deploy CampusMate to AgentCore Runtime

**CampusMate v4 (your agent, with memory, live in the cloud)**  ·  ⏱️ Time budget: **15 min theory + 40 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

Your CampusMate runs only while your notebook is open, and only for you. To serve hundreds of students at 2 AM it needs a home in the cloud.
**Amazon Bedrock AgentCore Runtime** is that home: you hand over your code, it gives you a secure endpoint, scales it, and keeps every student's conversation separate.

### What you'll do
- Create an AgentCore project with the AgentCore CLI
- Put **your** CampusMate inside: the tools (04) and memory (05), plus a 3-line wrapper
- Deploy it with one command, then **invoke** it
- Prove sessions and memory work in the cloud
- Clean up everything at the end

## Settings

✏️ Change `STUDENT_ID` to your own ID (the same one as notebook 05).

In [ ]:
# ---- Step 0: make sure the workshop libraries are in THIS notebook's Python (macOS, Windows, Linux) ----
import sys, subprocess, importlib, importlib.util
if sys.version_info < (3, 10):
    raise SystemExit(f"This notebook runs Python {sys.version.split()[0]}. Choose the .venv kernel (Python 3.10+) and run again.")
if any(importlib.util.find_spec(m) is None for m in ("strands", "strands_tools", "bedrock_agentcore", "mcp_proxy_for_aws", "boto3", "dotenv")):
    print("Installing the workshop libraries into", sys.executable, "(1-2 minutes, only the first time)...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                               "strands-agents==1.64.0", "strands-agents-tools==0.9.0", "bedrock-agentcore==1.25.4",
                               "mcp-proxy-for-aws==1.8.0", "boto3==1.43.126", "botocore==1.43.126",
                               "flask==3.1.3", "python-dotenv==1.2.4", "requests==2.34.2"])
        importlib.invalidate_caches()
    except subprocess.CalledProcessError:
        raise SystemExit("Could not install here. Choose the .venv kernel (VS Code: Select Kernel -> .venv; "
                         "Jupyter: start it from a terminal after activating .venv), then run this cell again.")
print("Libraries ready in:", sys.executable)

# Settings for this lab
import sys, json, time, uuid, pathlib, subprocess, shutil
import boto3

# ✏️ Put YOUR student ID here (the same one in every notebook). Lower-case letters and digits, e.g. priya07
STUDENT_ID = "student01"

REGION = "us-east-1"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"         # Amazon Nova 2 Lite
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"

PROJECT = f"campusmate{STUDENT_ID}"                    # your AgentCore project (created in notebook 05)
PROJECT_DIR = ROOT / PROJECT                           # the folder it lives in
AGENT_FILE = PROJECT_DIR / "app" / "agent" / "main.py" # your agent's code

def agentcore(*args, folder=None):
    """Run an AgentCore CLI command (the same one you would type in a terminal) and show its output."""
    process = subprocess.Popen([shutil.which("agentcore") or "agentcore", *args], cwd=folder or PROJECT_DIR,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in process.stdout:
        if "[SDK info]" not in line:
            print(line, end="")
    if process.wait() != 0:
        raise RuntimeError("The agentcore command failed. Read the messages above.")

print("Student:", STUDENT_ID, "| Project folder:", PROJECT_DIR)

## The prototype-to-production gap

| On your laptop (notebook 04) | In production |
|---|---|
| One user: you | Hundreds of students at once |
| Runs while the notebook is open | Always available, pay only when used |
| Uses your AWS keys | Gets its own AWS identity (a *role*) with just what it needs |
| `print()` | Logs, traces and metrics (Lab 9) |
| One conversation | Thousands of separate conversations |

🏫 **Analogy:** cooking for yourself vs running the college canteen at lunch hour.

## What AgentCore Runtime gives you

- **A separate little machine per conversation**: each *session* runs in its own isolated micro virtual machine (microVM).
- **A simple contract**: your code answers `POST /invocations`. The `BedrockAgentCoreApp` class does that for you; you only mark one
  function with `@app.entrypoint`.
- **Sessions**: every request carries a session ID (at least 33 characters). Same ID → same microVM → it still has the conversation.

We use the official **AgentCore CLI** (`agentcore`). It creates the project, the AWS role, the package and the Runtime for you.

## Step 1: create your AgentCore project

This makes a folder called `campusmate<your id>` next to `notebooks/`, with a starter agent inside. It takes about a minute.

(In a terminal this is `agentcore create --name campusmate<your id> --template agent-python-minimal`.)

In [ ]:
if PROJECT_DIR.exists():
    print("Your project already exists:", PROJECT_DIR)
else:
    agentcore("create", "--name", PROJECT, "--template", "agent-python-minimal", "--skip-git", folder=ROOT)

## Step 2: look at what the CLI made

The important files:
- `app/agent/main.py`: **your agent's code** (we replace it in Step 3)
- `app/agent/pyproject.toml`: the Python libraries your agent needs
- `agentcore/agentcore.json`: what your project contains (right now: one Runtime called `agent`)

Here is the starter `main.py`. Notice the 3 AgentCore lines: create the `app`, mark a function with `@app.entrypoint`, and `app.run()`.

In [ ]:
print(AGENT_FILE.read_text(encoding="utf-8"))

## Step 3: put YOUR CampusMate inside

The same tools from notebook 04 and the same memory from notebook 05. The only new part is at the bottom: the AgentCore wrapper.

- `invoke(payload, context)` runs for every message. `payload` is what the student sent; `context.session_id` says which conversation it belongs to.
- `MEMORY_ID` is filled in by AgentCore after you add a Memory to the project (Step 4).

In [ ]:
%%writefile {AGENT_FILE}
import json
import os
from pathlib import Path
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

DATA_DIR = Path(__file__).parent / "data"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"   # Amazon Nova 2 Lite
MEMORY_ID = os.getenv("MEMORY_STUDENTMEMORY_ID")   # AgentCore fills this in for you after `agentcore add memory`


# ---- Your tools from notebook 04 (unchanged) ----
@tool
def get_course_info(query: str) -> str:
    """Look up AIDD courses by course code (e.g. AIDD303) or a keyword in the title/description (e.g. docker).

    Args:
        query: A course code or a keyword.
    """
    q = query.lower()
    courses = json.loads((DATA_DIR / "courses.json").read_text(encoding="utf-8"))["courses"]
    hits = [c for c in courses if q == c["code"].lower() or q in (c["title"] + " " + c["description"]).lower()]
    return json.dumps(hits[:5]) if hits else json.dumps({"found": False, "codes": [c["code"] for c in courses]})


@tool
def get_campus_policy(topic: str) -> str:
    """Return CampusMate rules for a topic: attendance, internal_assessments, semester_end_exams, fees, leave,
    re_evaluation, library, hackathons. Always use this instead of guessing a rule.

    Args:
        topic: one of the topics above
    """
    policies = json.loads((DATA_DIR / "campus_policies.json").read_text(encoding="utf-8"))["policies"]
    return json.dumps(policies.get(topic.lower().strip(), {"found": False, "topics": list(policies)}))


# ---- Memory, like notebook 05 ----
def memory_for(session, student):
    """Save this conversation, and bring back what we already know about this student."""
    return AgentCoreMemorySessionManager(
        AgentCoreMemoryConfig(memory_id=MEMORY_ID, session_id=session, actor_id=student,
                              retrieval_config={"/users/{actorId}/facts": RetrievalConfig(top_k=5),
                                                "/summaries/{actorId}": RetrievalConfig(top_k=3)}),
        region_name="us-east-1")


# ---- NEW: the AgentCore Runtime wrapper ----
app = BedrockAgentCoreApp()
agents = {}   # one agent per conversation (session)


@app.entrypoint
def invoke(payload, context):
    student = payload.get("student_id", "AIDD-DEMO-001")
    session = context.session_id
    if session not in agents:
        agents[session] = Agent(
            model=BedrockModel(model_id=MODEL_ID, max_tokens=1200),
            tools=[get_course_info, get_campus_policy],
            session_manager=memory_for(session, student),
            system_prompt=f"You are CampusMate for AIDD students, talking to student {student}. "
                          "Use tools for courses and rules; never invent rules. Be brief.",
        )
    answer = agents[session](payload["prompt"])
    return {"result": str(answer)}


if __name__ == "__main__":
    app.run()

Your agent reads the campus files, so we copy `data/` into the project. Then we list the libraries it needs in the cloud.
We write them straight into `pyproject.toml` and pin Python 3.12, so packaging works the same on every laptop.

In [ ]:
# 1. Your agent reads the campus files, so copy them next to main.py
shutil.copytree(DATA_DIR, AGENT_FILE.parent / "data", dirs_exist_ok=True)

# 2. The libraries your agent needs in the cloud, written straight into pyproject.toml.
#    AgentCore installs exactly these when it packages your agent (this works the same on macOS and Windows).
(AGENT_FILE.parent / "pyproject.toml").write_text("""[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "campusmate-agent"
version = "0.1.0"
requires-python = ">=3.12,<3.13"
dependencies = [
    "aws-opentelemetry-distro ~= 0.21.0",
    "bedrock-agentcore == 1.25.4",
    "botocore[crt] ~= 1.43.0",
    "strands-agents == 1.64.0",
]

[tool.hatch.build.targets.wheel]
packages = ["."]
""", encoding="utf-8")

# 3. Run the agent on Python 3.12 in the cloud (the version we tested)
spec_file = PROJECT_DIR / "agentcore" / "agentcore.json"
spec = json.loads(spec_file.read_text(encoding="utf-8"))
for runtime in spec["runtimes"]:
    runtime["runtimeVersion"] = "PYTHON_3_12"
spec_file.write_text(json.dumps(spec, indent=2), encoding="utf-8")

deps = (AGENT_FILE.parent / "pyproject.toml").read_text(encoding="utf-8")
print("✅ strands-agents is in pyproject.toml" if "strands-agents" in deps else "❌ strands-agents missing")
print("Your agent folder now has:", sorted(p.name for p in AGENT_FILE.parent.iterdir() if not p.name.startswith(".")))

## Step 4: add a Memory to your project

(In a terminal: `agentcore add memory --name studentMemory --strategies SEMANTIC,SUMMARIZATION`.) This only updates your project file;
AWS creates the Memory when you deploy. The deployed agent gets its own Memory, separate from the one you made in notebook 05.

In [ ]:
if "studentMemory" in (PROJECT_DIR / "agentcore" / "agentcore.json").read_text(encoding="utf-8"):
    print("studentMemory is already in your project")
else:
    agentcore("add", "memory", "--name", "studentMemory", "--strategies", "SEMANTIC,SUMMARIZATION")

## Step 5: deploy

One command. Behind the scenes `agentcore deploy`:
1. creates an AWS **role** (an ID card) for your agent, allowed to call Bedrock, use its Memory and write logs,
2. packages your code and its libraries for the cloud's computers,
3. creates the **Memory** and the **AgentCore Runtime**.

The first deploy takes **3-5 minutes**. Read the messages as they appear.

In [ ]:
agentcore("deploy")

## What you deployed

The AgentCore CLI knows everything in your project. We ask it, so there are no IDs to copy.

In [ ]:
def deployed():
    """Ask the AgentCore CLI what your project has in AWS: {name: details}."""
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "status", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True, encoding="utf-8", errors="replace")
    return {r["name"]: r for r in json.loads(out.stdout)["resources"]}

def ask(prompt, session_id, student="AIDD-DEMO-001"):
    """Send one message to your deployed agent (agentcore invoke) and return its answer."""
    payload = json.dumps({"prompt": prompt, "student_id": student})
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "invoke", "--json", "--runtime", "agent",
                          "--session-id", session_id, "--payload", payload], cwd=PROJECT_DIR, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return json.loads(json.loads(out.stdout)["body"])["result"]

resources = deployed()
AGENT_ARN = resources["agent"]["arn"]
RUNTIME_ID = AGENT_ARN.split("/")[-1]
LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT"      # your agent's logs and traces
for name, r in resources.items():
    print(f"{r['resourceType']:14s} {name:16s} {r.get('arn', '')}")

## Step 6: talk to your deployed agent

`ask()` (defined above) uses `agentcore invoke`. The terminal version is:

`agentcore invoke --runtime agent --session-id <id> --payload '{"prompt": "..."}'`

**Windows:** quotes inside `--payload` behave differently in PowerShell and Command Prompt. Run that line in **Git Bash**, or just use `ask()` in this notebook.

In [ ]:
session_a = f"lab07-{uuid.uuid4()}"      # a session ID: 42 characters (Runtime needs at least 33)
print(ask("Hi, I'm Kiran. How is the Docker course assessed?", session_a))

## Step 7: prove sessions work

Same session again, then a brand-new session.

In [ ]:
print("Same session :", ask("What's my name?", session_a))
session_b = f"lab07-{uuid.uuid4()}"
print("New session  :", ask("What's my name?", session_b))

Session A remembered *Kiran*, session B did not. That memory lives **inside session A's microVM** and disappears when the session
ends (after about 15 idle minutes). Memory that lasts across sessions is next.

## Step 8: memory in the cloud

Tell the deployed CampusMate a fact as **AIDD-DEMO-001**, wait for long-term memory (30-90 seconds, in the background), then ask in a **new** session.

In [ ]:
STUDENT = "AIDD-DEMO-001"
MEMORY_ID = deployed()["studentMemory"]["arn"].split("/")[-1]      # the Memory AgentCore created for your project
print("Project memory:", MEMORY_ID)
print(ask("Hi! I'm in 3rd semester and I'm really interested in cloud computing.", f"lab07-A-{uuid.uuid4()}", STUDENT))

In [ ]:
memory_api = boto3.client("bedrock-agentcore", region_name=REGION)

def facts_about(student):
    r = memory_api.retrieve_memory_records(memoryId=MEMORY_ID, namespacePath=f"/users/{student}/facts",
                                           searchCriteria={"searchQuery": "semester and interests", "topK": 5})
    return [x["content"]["text"] for x in r.get("memoryRecordSummaries", [])]

for attempt in range(12):            # up to about 3 minutes
    found = facts_about(STUDENT)
    if any("cloud" in f.lower() for f in found):
        break
    print(f"  attempt {attempt + 1}: not there yet, waiting 15 s...")
    time.sleep(15)
print("Long-term facts:", found)
print(ask("Suggest one topic I should explore next. Use what you know about me.", f"lab07-B-{uuid.uuid4()}", STUDENT))

## ✍️ Your turn

Ask your deployed agent two questions in a **new** session: one about a course and one about a campus rule.

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
MY_SESSION = "TODO"   # TODO: make a new session ID, e.g. f"mine-{uuid.uuid4()}"

if MY_SESSION != "TODO":
    print(ask("TODO: a question about a course", MY_SESSION))
    print(ask("TODO: a question about a campus rule", MY_SESSION))
else:
    print("Skipped: set MY_SESSION first.")

## ✅ Checkpoint

You are on track if:
- Step 1 created the folder `campusmate<your id>`
- Step 3 printed `✅ strands-agents is in pyproject.toml`
- Step 5 ended with `deployed project ...`
- Session A remembered Kiran; session B did not
- Step 8: a new session's suggestion mentions cloud

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| `'agentcore' is not recognized` / `FileNotFoundError: agentcore` | The AgentCore CLI isn't installed | Run `npm install -g @aws/agentcore` (needs Node.js 20+), then restart Jupyter |
| `uv: command not found` | uv isn't installed (the AgentCore CLI needs it) | Install uv (SETUP.md), then restart Jupyter |
| Deploy fails with a CDK / bootstrap message | The account needs a one-time CDK setup | Ask your instructor (they run `cdk bootstrap` once per account) |
| `ValidationException ... runtimeSessionId` | Session ID shorter than 33 characters | Use `f"lab05-{uuid.uuid4()}"` |
| Project name error | Project names allow only letters and digits | Use a STUDENT_ID with lower-case letters and digits only |
| `No module named 'strands'` when you invoke | The agent was deployed before Step 3 wrote its libraries | Run Step 3 again, then deploy again |
| `ModuleNotFoundError: No module named 'strands'` in a cell | The notebook isn't using `.venv` | Run the Settings cell again, or VS Code: Select Kernel → `.venv` |

## 🧹 Finish: delete everything you created in AWS

This deletes your AgentCore project (Runtime + its Memory) **and** the Memory you made in notebook 05. It works the same on macOS and Windows.

1. Check `STUDENT_ID` in the Settings cell is **yours**.
2. Change `CONFIRM = False` to `CONFIRM = True`.
3. Run the cell. It starts the delete and ends with **PASS** in a few seconds. AWS finishes it in 10-20 minutes.

🎉 **You built CampusMate:** a model with tools (04), memory (05), a report card (06), and a home in the cloud (07).

In [ ]:
# 🧹 Delete everything you created in AWS: the AgentCore project (Runtime + Memory) and your notebook-05 Memory
CONFIRM = False      # ✏️ change to True, then run this cell

from botocore.exceptions import ClientError
from bedrock_agentcore.memory import MemoryClient

STACK = f"AgentCore-campusmate{STUDENT_ID}-default"
cfn = boto3.client("cloudformation", region_name=REGION)

def stack_exists():
    try:
        return cfn.describe_stacks(StackName=STACK)["Stacks"][0]["StackStatus"] != "DELETE_COMPLETE"
    except ClientError:
        return False

if not CONFIRM:
    print(f"Nothing deleted. Set CONFIRM = True to delete {STACK} and your notebook-05 Memory.")
else:
    if stack_exists():
        cfn.delete_stack(StackName=STACK)           # AWS finishes this in the background (10-20 minutes)
        print("Runtime + project Memory: deleting now (10-20 minutes, you can close the notebook)")
    else:
        print("Runtime + project Memory: gone ✅")

    memory_client = MemoryClient(region_name=REGION)
    mine = [m["id"] for m in memory_client.list_memories()
            if m.get("id", "").startswith(f"campusmate_mem_{STUDENT_ID}-") and m.get("status") != "DELETING"]
    for memory_id in mine:
        memory_client.delete_memory(memory_id)      # also finishes in the background
    print("Notebook-05 Memory:", f"deleting {mine}" if mine else "nothing left to delete ✅")
    print("PASS: cleanup started. Run this cell again later; it should say 'gone' twice.")